# THIS IS THE MAIN ONE
... referenced in the thesis.

### Needs `01-Prep-GEPA.ipynb` to be executed first

Reads from:
- `32B_H200_ynorm.json`

# Preperations

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
GEPA_RUN  = "32B_H200"
GEPA_PATH = Path(f"GEPA_runs/{GEPA_RUN}")

## Import Data

In [3]:
gs_run_ynorm = pd.read_json(f"{GEPA_RUN}_ynorm.json")
# year_scope_df       = pd.read_json("year_scope_df.json")

CATEGORIES = ["value", "unit"]

In [4]:
CSV_LIST = sorted(
    [p for p in GEPA_PATH.glob("*.csv") if p.stem.isdigit()],
    key=lambda p: int(p.stem)
)

VARIANTS = [i for i in range(len(CSV_LIST))]
# e.g. [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29]

In [5]:
eval_log = pd.read_csv(GEPA_PATH / "eval_log.csv")

# The last generation = rows after the final occurrence of run==0
last_gen_start = eval_log[eval_log["run"] == 0].index[-1]
last_gen = eval_log.loc[last_gen_start:].reset_index(drop=True)

BEST_VARIANTS = last_gen[last_gen["is_best"] == True]["run"].tolist()
print(f"Best variants from {GEPA_RUN}: ", BEST_VARIANTS)

Best variants from 32B_H200:  [0, 3, 4, 5, 6, 15]


#### Cleanup Import

In [6]:
pipeline_cols = (
    [f"value_{v}" for v in VARIANTS] +
    [f"unit_{v}"  for v in VARIANTS] +
    [f"label_{v}" for v in VARIANTS]
)
    
for col in pipeline_cols:
    gs_run_ynorm[col] = gs_run_ynorm[col].apply(
    lambda x: np.nan if x is None else x
)

## Defining Evaluation Functions

In [7]:
def check_hit(row, extraction_col, gs_col) -> bool:
    if isinstance(row[extraction_col], list): # Proceedes into nested code only if it is a list (and therefore has values in it)
        return row[gs_col] in row[extraction_col]
    else:
        return pd.isna(row[gs_col]) # Only executed if the extraction found nothing ("NaN") and returns True if there is really nothing in the Gold-Standard

def check_hit_exactly(row, extraction_col, gs_col) -> bool:
    # Preparing gold-standard value(s) for comparison
    gs_val  = row[gs_col]
    if isinstance(gs_val,  list):
        gs_set = set(gs_val)
    elif not pd.isna(gs_val):
        gs_set = {gs_val}
    else:
        gs_set = set()
    
    # Preparing extracted value(s) for comparison
    ext_val = row[extraction_col]
    if isinstance(ext_val, list):
        ext_set = set(ext_val)
    elif not pd.isna(ext_val):
        ext_set = {ext_val}
    else:
        ext_set = set()
    
    # Compare value(s), as there could be more than one (e.g., Allianz) or just one (mostly)
    return gs_set == ext_set

In [8]:
def eval_intern(source, result_basis, exact) -> pd.DataFrame:
    result = result_basis.copy()
    if exact:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit_exactly,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
    else:
        for cat in CATEGORIES: # "value" then "unit" comparison
            for v in VARIANTS: # "t1" -> "t2" -> "m1" ...
                result[f"{v}_{cat}_hit"] = source.apply(
                    check_hit,                  # Apply check for every row
                    args=(f"{cat}_{v}", cat),   # Forwarding arguments from for-loop
                    axis=1)                     # enables row-wise not column-wise applying (that would be axis=0)
                
    return result

def eval(source, exact) -> pd.DataFrame:
    
    # Creating smaller DataFrame to just see the True/False mapping
    gs_eval  = eval_intern(source, source[["report_name","year","scope"]], exact)
    
    # Second run, this time the returned DF is an extended form of the source DF (for further analysis)
    in_place = eval_intern(source, source, exact)
    
    return gs_eval, in_place



def print_eval(gs_eval) -> None: 
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in VARIANTS:
            col = gs_eval[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean()*100,2) #Percent
            print(f"{v}: True = {true_count} || False = {false_count} || Quota = {quota}%")
        print()

# Evaluations

## YNORM

### Exact

In [9]:
ynorm_exact, _ = eval(gs_run_ynorm, True)

print_eval(ynorm_exact)

### value ###
0: True = 1276 || False = 86 || Quota = 93.69%
1: True = 1241 || False = 121 || Quota = 91.12%
2: True = 1179 || False = 183 || Quota = 86.56%
3: True = 1246 || False = 116 || Quota = 91.48%
4: True = 1268 || False = 94 || Quota = 93.1%
5: True = 1276 || False = 86 || Quota = 93.69%
6: True = 1273 || False = 89 || Quota = 93.47%
7: True = 1290 || False = 72 || Quota = 94.71%
8: True = 1274 || False = 88 || Quota = 93.54%
9: True = 1284 || False = 78 || Quota = 94.27%
10: True = 1270 || False = 92 || Quota = 93.25%
11: True = 1271 || False = 91 || Quota = 93.32%
12: True = 1211 || False = 151 || Quota = 88.91%
13: True = 1272 || False = 90 || Quota = 93.39%
14: True = 1246 || False = 116 || Quota = 91.48%
15: True = 1290 || False = 72 || Quota = 94.71%
16: True = 1259 || False = 103 || Quota = 92.44%
17: True = 1274 || False = 88 || Quota = 93.54%
18: True = 1070 || False = 292 || Quota = 78.56%
19: True = 1277 || False = 85 || Quota = 93.76%
20: True = 1101 || False = 261

# Best-Variant Comparison (is_best == True)

In [10]:
def eval_best(source, exact) -> pd.DataFrame:
    """Like eval(), but restricted to BEST_VARIANTS only."""
    result_basis = source[["report_name", "year", "scope"]]
    result = result_basis.copy()
    check_fn = check_hit_exactly if exact else check_hit

    for cat in CATEGORIES:
        for v in BEST_VARIANTS:
            result[f"{v}_{cat}_hit"] = source.apply(
                check_fn, args=(f"{cat}_{v}", cat), axis=1
            )
    return result


def print_eval_best(gs_eval_best) -> None:
    for cat in CATEGORIES:
        print(f"### {cat} ###")
        for v in BEST_VARIANTS:
            col = gs_eval_best[f"{v}_{cat}_hit"]
            true_count  = col.value_counts()[True]
            false_count = col.value_counts()[False]
            quota       = round(col.mean() * 100, 2)
            score_row   = last_gen[last_gen["run"] == v].iloc[0]
            print(f"run {v:>2} (GEPA score {score_row['score']:.4f}): "
                  f"True={true_count} | False={false_count} | Quota={quota}%")
        print()

In [11]:

best_ynorm      = eval_best(gs_run_ynorm, exact=False)

print_eval_best(best_ynorm)

### value ###
run  0 (GEPA score 0.8968): True=1292 | False=70 | Quota=94.86%
run  3 (GEPA score 0.8996): True=1267 | False=95 | Quota=93.02%
run  4 (GEPA score 0.9073): True=1296 | False=66 | Quota=95.15%
run  5 (GEPA score 0.9192): True=1285 | False=77 | Quota=94.35%
run  6 (GEPA score 0.9215): True=1284 | False=78 | Quota=94.27%
run 15 (GEPA score 0.9307): True=1296 | False=66 | Quota=95.15%

### unit ###
run  0 (GEPA score 0.8968): True=1199 | False=163 | Quota=88.03%
run  3 (GEPA score 0.8996): True=1184 | False=178 | Quota=86.93%
run  4 (GEPA score 0.9073): True=1182 | False=180 | Quota=86.78%
run  5 (GEPA score 0.9192): True=1196 | False=166 | Quota=87.81%
run  6 (GEPA score 0.9215): True=1208 | False=154 | Quota=88.69%
run 15 (GEPA score 0.9307): True=1212 | False=150 | Quota=88.99%



In [12]:
rows = []
for v in BEST_VARIANTS:
    score_row = last_gen[last_gen["run"] == v].iloc[0]
    for cat in CATEGORIES:
        col = best_ynorm[f"{v}_{cat}_hit"]
        rows.append({
            "run": v,
            "gepa_score": round(score_row["score"], 6),
            "category": cat,
            "eval_type": "ynorm",
            "quota": round(col.mean() * 100, 2),
            "hits": col.value_counts()[True],
        })

best_summary = pd.DataFrame(rows)
best_summary.pivot_table(
    index=["run", "gepa_score"],
    columns=["eval_type", "category"],
    values="quota"
).round(2)

eval_type       ynorm       
category         unit  value
run gepa_score              
0   0.896843    88.03  94.86
3   0.899617    86.93  93.02
4   0.907280    86.78  95.15
5   0.919157    87.81  94.35
6   0.921456    88.69  94.27
15  0.930651    88.99  95.15

In [13]:
best_summary.to_csv(f"summary_{GEPA_RUN}")

#### Looking into the console log from the run `43321517_out.log`:

Just looking at 'is_best' from `eval_log.csv`, which was my own creation, seems to be misleading, due to the logic on how GEPA compares older prompts to the newly tested ones. Therefore:

- Runs 0 and 1 are both with the initial prompt `Query0_Extraction.txt`
- Runs `2/7/9/10/12/14/16/18/20/22` are mutated prompts that got not validated in the second run and were therefore omitted.
- Runs `4/5/6/8/11/13/15/17/19/21/23` were all run with the identical prompt, which was the prompt that is given back as the 'best_candidate' by GEPA in `oa_result.txt`

This aligns with the runs pointed out two cells above:
- Run 3 was performed on a 'subsample' by GEPA out of which a new prompt was proposed
- Run 4 was also performed on that 'subsample' but with the new prompt and beat the old prompt
- Run 5 validated the new prompt on the entire GEPA_Train_Set given to GEPA and beat the old prompt there as well. Therefore, it was considered a new, validated better prompt.

Run 5 was also the last run to achieve this, as the other runs used this exact prompt from run 4 and 5, see above.